# LightGBM vs LightGBMLSS CUDA Benchmark Analysis

This notebook combines benchmark outputs from:
- `synthetic/lightgbm_cuda_benchmark.py`
- `synthetic/lightgbmlss_cuda_benchmark.py`

It provides a unified view of runtime behavior across CPU and CUDA, including an overlaid training-time plot for both methods.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.figsize"] = (12, 7)

RESULTS = {
    "LightGBM": Path("../results/synthetic/lightgbm_cuda_benchmark"),
    "LightGBMLSS": Path("../results/synthetic/lightgbmlss_cuda_benchmark"),
}

COMBINED_PATH = Path("../results/synthetic/combined_lightgbm_vs_lightgbmlss_cuda_benchmark.csv")

In [ ]:
def load_runs(model_name: str, results_dir: Path) -> pd.DataFrame:
    csv_files = sorted(results_dir.glob("benchmark_*.csv"))
    print(f"[{model_name}] found {len(csv_files)} files in {results_dir}")
    if not csv_files:
        raise FileNotFoundError(f"No benchmark CSVs found for {model_name} in {results_dir}")

    frames = []
    for path in csv_files:
        frame = pd.read_csv(path)
        frame["source_file"] = path.name
        frame["model"] = model_name
        frames.append(frame)

    runs = pd.concat(frames, ignore_index=True)
    runs["error"] = runs.get("error", "").fillna("")
    if "stabilization" in runs.columns:
        runs["stabilization"] = runs["stabilization"].fillna("None")

    numeric_cols = [
        "n_samples",
        "n_features",
        "n_informative",
        "train_fraction",
        "num_boost_round",
        "early_stopping_rounds",
        "repeat_idx",
        "seed",
        "start_value_seconds",
        "train_seconds",
        "predict_seconds",
        "best_iteration",
        "speedup_vs_cpu",
        "num_threads",
    ]
    for col in numeric_cols:
        if col in runs.columns:
            runs[col] = pd.to_numeric(runs[col], errors="coerce")

    return runs

In [ ]:
all_runs = []
for model_name, results_dir in RESULTS.items():
    model_runs = load_runs(model_name, results_dir)
    all_runs.append(model_runs)

runs = pd.concat(all_runs, ignore_index=True)
display(runs.head())

print("Total rows:", len(runs))
print("Models:", sorted(runs["model"].dropna().unique().tolist()))
print("Devices:", sorted(runs["device"].dropna().unique().tolist()))
print("Statuses:\
", runs["status"].value_counts(dropna=False))

ok_runs = runs.loc[runs["status"] == "ok"].copy()
failed_runs = runs.loc[runs["status"] != "ok"].copy()

print(f"Successful rows: {len(ok_runs)}")
print(f"Failed/skipped rows: {len(failed_runs)}")
if not failed_runs.empty:
    display(
        failed_runs[[
            "model", "source_file", "n_samples", "seed",
            "device", "status", "probe_status", "error"
        ]].sort_values(["model", "n_samples", "seed", "device"])
    )

COMBINED_PATH.parent.mkdir(parents=True, exist_ok=True)
ok_runs.to_csv(COMBINED_PATH, index=False)
print(f"Wrote combined successful runs to {COMBINED_PATH}")

In [ ]:
summary = (
    ok_runs.groupby(["model", "n_samples", "device"], as_index=False)
    .agg(
        repeats=("seed", "nunique"),
        train_mean=("train_seconds", "mean"),
        train_std=("train_seconds", "std"),
        predict_mean=("predict_seconds", "mean"),
        predict_std=("predict_seconds", "std"),
        best_iteration_mean=("best_iteration", "mean"),
        best_iteration_std=("best_iteration", "std"),
    )
    .sort_values(["model", "n_samples", "device"])
)

display(summary)

In [ ]:
# Combined training-time plot: both methods on one figure
fig, ax = plt.subplots(figsize=(14, 8))

sns.lineplot(
    data=ok_runs,
    x="n_samples",
    y="train_seconds",
    hue="device",
    style="model",
    markers=True,
    dashes=True,
    estimator="mean",
    errorbar="sd",
    ax=ax,
)

ax.set_title("Training Time by Sample Size (LightGBM vs LightGBMLSS)")
ax.set_xlabel("n_samples")
ax.set_ylabel("train_seconds")
ax.set_xscale("log")
ax.legend(title="Device / Model", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
# Optional companion view for prediction time
fig, ax = plt.subplots(figsize=(14, 8))

sns.lineplot(
    data=ok_runs,
    x="n_samples",
    y="predict_seconds",
    hue="device",
    style="model",
    markers=True,
    dashes=True,
    estimator="mean",
    errorbar="sd",
    ax=ax,
)

ax.set_title("Prediction Time by Sample Size (LightGBM vs LightGBMLSS)")
ax.set_xlabel("n_samples")
ax.set_ylabel("predict_seconds")
ax.set_xscale("log")
ax.legend(title="Device / Model", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
# Per-model CPU/CUDA speedup comparison
paired = (
    ok_runs.pivot_table(
        index=["model", "n_samples", "seed"],
        columns="device",
        values=["train_seconds", "predict_seconds", "best_iteration"],
        aggfunc="first",
    )
    .sort_index()
)
paired.columns = [f"{metric}_{device}" for metric, device in paired.columns]
paired = paired.reset_index()

paired["train_speedup_cpu_over_cuda"] = paired["train_seconds_cpu"] / paired["train_seconds_cuda"]
paired["predict_speedup_cpu_over_cuda"] = paired["predict_seconds_cpu"] / paired["predict_seconds_cuda"]
paired["best_iteration_delta_cuda_minus_cpu"] = paired["best_iteration_cuda"] - paired["best_iteration_cpu"]

display(paired.head())

speedup_summary = (
    paired.groupby(["model", "n_samples"], as_index=False)
    .agg(
        repeats=("seed", "nunique"),
        train_speedup_mean=("train_speedup_cpu_over_cuda", "mean"),
        train_speedup_std=("train_speedup_cpu_over_cuda", "std"),
        predict_speedup_mean=("predict_speedup_cpu_over_cuda", "mean"),
        predict_speedup_std=("predict_speedup_cpu_over_cuda", "std"),
        best_iteration_delta_mean=("best_iteration_delta_cuda_minus_cpu", "mean"),
    )
    .sort_values(["model", "n_samples"])
)
display(speedup_summary)

## Notes

- The combined training-time plot (Cell 7) overlays LightGBM and LightGBMLSS on the same axes.
- If `train_speedup_cpu_over_cuda > 1`, CUDA is faster for training.
- `best_iteration` is a convergence proxy and not a direct forecast-quality metric.
- To compare predictive quality, extend benchmark scripts to log RMSE, NLL, CRPS, or quantile loss on the same validation split.